<a href="https://colab.research.google.com/github/sudinisreeja/Visa-Status-Prediction-/blob/main/Milestone_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import numpy as np

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [27]:
df = pd.read_csv('/content/Visa Status Prediction Cleaned Dataset.csv')
print(df.head())

   S.NO CASE_STATUS                                  EMPLOYER_NAME  \
0     1   CERTIFIED                                INFOSYS LIMITED   
1     2   CERTIFIED                                INFOSYS LIMITED   
2     3   CERTIFIED                            KRONOS INCORPORATED   
3     4   CERTIFIED  UNIVERSITY OF MIAMI-MILLER SCHOOL OF MEDICINE   
4     5   CERTIFIED               LARSEN & TOUBRO INFOTECH LIMITED   

                          SOC_NAME                 JOB_TITLE  AGE GENDER  \
0        COMPUTER SYSTEMS ANALYSTS      TECHNOLOGY LEAD - US   70      F   
1  COMPUTER OCCUPATIONS, ALL OTHER  TECHNICAL TEST LEAD - US   69      F   
2          DATABASE ADMINISTRATORS    DATABASE ADMINISTRATOR   54      F   
3    BIOCHEMISTS AND BIOPHYSICISTS  SR. RESEARCH ASSOCIATE 1   55      M   
4   BUSINESS INTELLIGENCE ANALYSTS          BUSINESS ANALYST   49      M   

  FULL_TIME_POSITION  PREVAILING_WAGE  YEAR                   WORKSITE  \
0                  Y          84032.0  2015     

In [29]:
df['DECISION DATE'] = pd.to_datetime(df['DECISION DATE'])
df['APPLICATION DATE'] = pd.to_datetime(df['APPLICATION DATE'])
df["processing_days"] = (df["DECISION DATE"] - df["APPLICATION DATE"]).dt.days

In [30]:
df['APPLICATION DATE'] = pd.to_datetime(df['APPLICATION DATE'])

In [43]:
df_encoded = pd.get_dummies(df, columns=['VISA TYPE', 'EDUCATION', 'CASE_STATUS', 'EMPLOYER_NAME', 'SOC_NAME', 'JOB_TITLE', 'WORKSITE', 'GENDER', 'FULL_TIME_POSITION'], drop_first=True)

In [48]:
df_encoded.dropna(subset=['processing_days'], inplace=True)
X = df_encoded.drop(
    columns=["processing_days", "APPLICATION DATE", "DECISION DATE"]
)
y = df_encoded["processing_days"]

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [50]:
model= LinearRegression()

In [51]:
model.fit(X_train, y_train)# this is where the learning  happens

LinearRegression()

In [52]:
y_pred = model.predict(X_test)

In [53]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 49.225621076105874
RMSE: 87.09438156542427


In [54]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [55]:
rf_model = RandomForestRegressor(random_state=42)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

In [56]:
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)

Random Forest MAE: 42.13612903225807
Random Forest RMSE: 82.69654484337455


In [57]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [58]:
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=2,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

In [59]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=2, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100, 200]},
             scoring='neg_mean_absolute_error')

In [60]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}


In [61]:
best_rf = grid_search.best_estimator_

best_predictions = best_rf.predict(X_test)

In [62]:
best_mae = mean_absolute_error(y_test, best_predictions)
best_rmse = np.sqrt(mean_squared_error(y_test, best_predictions))

print("Tuned Random Forest MAE:", best_mae)
print("Tuned Random Forest RMSE:", best_rmse)

Tuned Random Forest MAE: 42.41291967006212
Tuned Random Forest RMSE: 82.5917940765196
